# PagedAttention: Give Each Sequence a Block Table

Prerequisites: KV caching, stable softmax, dictionaries, and reference ownership.
Allow 60-90 minutes. Build a bounded cache, fork a shared prefix, append safely,
and verify attention still sees the same sequence as a contiguous cache.

## Why reservation wastes space

Output length is unknown when a request arrives. Reserving its maximum contiguous
cache leaves unused space; requiring growth into adjacent storage constrains
placement. Fixed-size KV blocks let requests grow using any available block.
A table maps each sequence's logical blocks to physical storage.

For block size B and logical token position t:

```text
logical_block = t // B
offset = t % B
physical_block = sequence_table[logical_block]
entry = physical_blocks[physical_block][offset]
```

With B=2 and table `[0,2,1]`, token position 5 lives in physical block 1, slot 1.
Sorting the table would scramble context. Paging changes addresses, not the
attention equation or which tokens the query may use.

## Follow one shared prefix

1. A caches three tokens: physical block 0 holds tokens 1,2; block 1 holds token 3.
2. Fork B from A. Both tables are `[0,1]`; each block has two owners. No copy yet.
3. B appends token 4. Block 1 is partial and shared, so copy it to free block 2
   and append there. B's table becomes `[0,2]`. A's history remains unchanged.
4. A finishes. Its reference to block 0 disappears, but B still owns that block.
   Block 1 reaches zero owners and becomes free.
5. B appends token 5. Reuse block 1. B's table becomes `[0,2,1]`.

A reference count records how many sequence tables own a physical block.
Copy-on-write means creating a private copy before modifying shared storage.
Forking must copy the list of block IDs; sharing the list object itself would
let one sequence change the other's mapping.

## Make out-of-memory behavior explicit

The pool has a fixed number of blocks. An append requiring a new block can fail.
Reserve the block before changing a table or reference count. If allocation
fails, raise `MemoryError` with both histories intact. A serving scheduler can
then defer, reject, or preempt work according to a separate policy.

Appending to a full shared block allocates a new block without copying the full
one. Appending to a partial shared block needs a copy. Once only one owner
remains, that partial block can be extended in place.

## Implement the cache and attention

Use dictionaries for block tables, block contents, and reference counts. Choose
the lowest free physical ID to make test traces reproducible. Each token stores
a scalar `(key,value)` pair so address translation is easy to inspect.

For scalar query q, calculate `scores = q*keys`, then stable softmax and the
weighted sum of values. All cached positions are visible at this decode step;
head dimension is 1, so the attention scale is 1. This reference materializes
logical entries for comparison. It does not implement a fused paged GPU kernel.

The `stats` contract counts physical storage once even if shared. For 9 unshared
tokens and B=4, 3 blocks allocate 12 slots and waste 3 inside occupied blocks.
Unallocated blocks are capacity, not internal fragmentation. Real blocks also
have metadata and scheduling costs. Sharing can further reduce duplicate prefix
storage; it does not make arbitrary contexts free.

## Verification and solutions

The grader checks non-contiguous addresses, reference counts through observable
release behavior, branch isolation, block reuse, immutable table snapshots, and
atomic failure during ordinary growth and copy-on-write. Decode is compared
against contiguous attention for positive, negative, zero, and large queries.

In the notebook, increase block size while keeping request lengths fixed.
Predict internal waste first. Then compare shared and duplicated prefixes.
Smaller blocks reduce tail waste but create more table entries and management
work; this simulator cannot measure GPU throughput or select the best production
block size. Paged allocation and continuous batching are related but distinct:
one manages memory, the other decides which requests execute together.

**Exit check:** Explain when a copy is necessary, when a block can be freed, and
why byte savings alone do not establish a serving throughput improvement.

Primary source: [Kwon et al., PagedAttention and vLLM](https://arxiv.org/abs/2309.06180).


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""A bounded paged KV cache with shared prefixes and copy-on-write.
One logical token holds a scalar key and scalar value. This makes address
translation visible without tensor indexing noise; it is not a GPU kernel.
"""
import numpy as np

class PagedCache:
    def __init__(self, block_size=2, total_blocks=4):
        """Positive integer sizes; invalid sizes raise ValueError.
        Store sequence block tables, physical blocks, and reference counts.
        """
        raise NotImplementedError

    def append(self, sequence, key, value):
        """Create sequence if needed; append finite scalar K/V.
        Allocate the lowest free block ID when necessary. If the last block is
        partial and shared, copy it before writing (copy-on-write).
        If no free block exists raise MemoryError WITHOUT changing any state.
        """
        raise NotImplementedError

    def fork(self, source, target):
        """Share source's blocks with new target. No physical copy yet.
        Missing source: KeyError. Existing target: ValueError. No mutation on error.
        """
        raise NotImplementedError

    def release(self, sequence):
        """Decrement references; free blocks only when no sequence owns them.
        Unknown sequence: KeyError. Releasing one branch must preserve the other.
        """
        raise NotImplementedError

    def read(self, sequence):
        """Logical ordered list of (key,value) pairs; unknown sequence: KeyError."""
        raise NotImplementedError

    def table(self, sequence):
        """Copy of logical->physical block IDs; unknown sequence: KeyError."""
        raise NotImplementedError

    def stats(self):
        """Return used_blocks, free_blocks, wasted_slots (in occupied blocks).
        Shared physical blocks count once. Unallocated blocks are not waste.
        """
        raise NotImplementedError

def paged_decode(query, cache, sequence):
    """Scalar query: stable softmax(query * cached_keys) dot cached_values.
    Read through the block table. All cached tokens are visible at this decode
    position. D=1, so sqrt(D)=1. Must match a contiguous-cache computation.
    """
    raise NotImplementedError


In [ ]:
import numpy as np
_c = PagedCache(block_size=2, total_blocks=4)
_c.append('a', 1, 10)
_c.append('a', 2, 20)
_c.append('b', 9, 90)
_c.append('a', 3, 30)
assert _c.table('a') == [0, 2], 'logical order differs from physical adjacency'
_c.fork('a', 'branch')
assert _c.stats() == {'used_blocks': 3, 'free_blocks': 1, 'wasted_slots': 2}
_c.append('branch', 4, 40)
assert _c.table('branch') == [0, 3], 'copy partial shared block before append'
assert _c.read('a') == [(1., 10.), (2., 20.), (3., 30.)]
assert _c.read('branch')[-1] == (4., 40.)
for _seq in ['a', 'branch', 'b']:
    _entries = np.array(_c.read(_seq))
    for _q in [0., 1., -2., 1000.]:
        _s = _q * _entries[:, 0]
        _p = np.exp(_s - _s.max()); _p /= _p.sum()
        assert abs(paged_decode(_q, _c, _seq) - _p @ _entries[:, 1]) < 1e-10
_before = (_c.read('branch'), _c.table('branch'), _c.stats())
try:
    _c.append('branch', 5, 50)
except MemoryError:
    pass
else:
    raise AssertionError('full pool must reject growth')
assert (_c.read('branch'), _c.table('branch'), _c.stats()) == _before
_c.release('a')
assert _c.read('branch') == [(1., 10.), (2., 20.), (3., 30.), (4., 40.)]
_c.append('branch', 5, 50)
assert _c.table('branch') == [0, 3, 2], 'reuse the freed block'
_c.release('b'); _c.release('branch')
assert _c.stats() == {'used_blocks': 0, 'free_blocks': 4, 'wasted_slots': 0}
# Copy-on-write failure must also leave both owners untouched.
_c = PagedCache(2, 1); _c.append('a', 1, 2); _c.fork('a', 'b')
try:
    _c.append('b', 3, 4)
except MemoryError:
    pass
else:
    raise AssertionError('shared partial block needs a free copy')
assert _c.read('a') == _c.read('b') == [(1., 2.)]
_c.release('a'); _c.append('b', 3, 4)
assert _c.read('b') == [(1., 2.), (3., 4.)], 'last owner may mutate in place'
_t = _c.table('b'); _t.clear()
assert _c.table('b') == [0], 'table() must not expose mutable ownership state'
for _operation, _error in [(lambda: _c.fork('b', 'b'), ValueError), (lambda: _c.release('missing'), KeyError), (lambda: PagedCache(0, 1), ValueError)]:
    try:
        _operation()
    except _error:
        pass
    else:
        raise AssertionError('invalid operation must fail')
print('Paged decode, sharing, copy-on-write, OOM atomicity, and block reclamation passed.')


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import numpy as np

class PagedCache:
    def __init__(self, block_size=2, total_blocks=4):
        if block_size <= 0 or total_blocks <= 0:
            raise ValueError('positive sizes required')
        self.block_size, self.total_blocks = block_size, total_blocks
        self.tables, self.blocks, self.refs = {}, {}, {}

    def append(self, sequence, key, value):
        ids = self.tables.get(sequence, [])
        partial = bool(ids) and len(self.blocks[ids[-1]]) < self.block_size
        shared = partial and self.refs[ids[-1]] > 1
        if not partial or shared:
            free = sorted(set(range(self.total_blocks)) - self.blocks.keys())
            if not free:
                raise MemoryError('KV block pool exhausted')
            # Reserve before changing a table or reference count: OOM is atomic.
            new = free[0]
            self.blocks[new] = list(self.blocks[ids[-1]]) if shared else []
            self.refs[new] = 1
            if shared:
                self.refs[ids[-1]] -= 1
                ids[-1] = new
            else:
                ids.append(new)
            self.tables[sequence] = ids
        self.blocks[ids[-1]].append((float(key), float(value)))

    def fork(self, source, target):
        ids = self.tables[source]
        if target in self.tables:
            raise ValueError('target already exists')
        self.tables[target] = list(ids)
        for block in ids:
            self.refs[block] += 1

    def release(self, sequence):
        for block in self.tables.pop(sequence):
            self.refs[block] -= 1
            if self.refs[block] == 0:
                del self.refs[block]
                del self.blocks[block]

    def read(self, sequence):
        return [kv for block in self.tables[sequence] for kv in self.blocks[block]]

    def table(self, sequence):
        return list(self.tables[sequence])

    def stats(self):
        used = len(self.blocks)
        return {'used_blocks': used, 'free_blocks': self.total_blocks - used,
                'wasted_slots': sum(self.block_size - len(b) for b in self.blocks.values())}

def paged_decode(query, cache, sequence):
    entries = np.array(cache.read(sequence))
    scores = query * entries[:, 0]
    p = np.exp(scores - scores.max())
    return float((p / p.sum()) @ entries[:, 1])


In [ ]:
import numpy as np
_c = PagedCache(block_size=2, total_blocks=4)
_c.append('a', 1, 10)
_c.append('a', 2, 20)
_c.append('b', 9, 90)
_c.append('a', 3, 30)
assert _c.table('a') == [0, 2], 'logical order differs from physical adjacency'
_c.fork('a', 'branch')
assert _c.stats() == {'used_blocks': 3, 'free_blocks': 1, 'wasted_slots': 2}
_c.append('branch', 4, 40)
assert _c.table('branch') == [0, 3], 'copy partial shared block before append'
assert _c.read('a') == [(1., 10.), (2., 20.), (3., 30.)]
assert _c.read('branch')[-1] == (4., 40.)
for _seq in ['a', 'branch', 'b']:
    _entries = np.array(_c.read(_seq))
    for _q in [0., 1., -2., 1000.]:
        _s = _q * _entries[:, 0]
        _p = np.exp(_s - _s.max()); _p /= _p.sum()
        assert abs(paged_decode(_q, _c, _seq) - _p @ _entries[:, 1]) < 1e-10
_before = (_c.read('branch'), _c.table('branch'), _c.stats())
try:
    _c.append('branch', 5, 50)
except MemoryError:
    pass
else:
    raise AssertionError('full pool must reject growth')
assert (_c.read('branch'), _c.table('branch'), _c.stats()) == _before
_c.release('a')
assert _c.read('branch') == [(1., 10.), (2., 20.), (3., 30.), (4., 40.)]
_c.append('branch', 5, 50)
assert _c.table('branch') == [0, 3, 2], 'reuse the freed block'
_c.release('b'); _c.release('branch')
assert _c.stats() == {'used_blocks': 0, 'free_blocks': 4, 'wasted_slots': 0}
# Copy-on-write failure must also leave both owners untouched.
_c = PagedCache(2, 1); _c.append('a', 1, 2); _c.fork('a', 'b')
try:
    _c.append('b', 3, 4)
except MemoryError:
    pass
else:
    raise AssertionError('shared partial block needs a free copy')
assert _c.read('a') == _c.read('b') == [(1., 2.)]
_c.release('a'); _c.append('b', 3, 4)
assert _c.read('b') == [(1., 2.), (3., 4.)], 'last owner may mutate in place'
_t = _c.table('b'); _t.clear()
assert _c.table('b') == [0], 'table() must not expose mutable ownership state'
for _operation, _error in [(lambda: _c.fork('b', 'b'), ValueError), (lambda: _c.release('missing'), KeyError), (lambda: PagedCache(0, 1), ValueError)]:
    try:
        _operation()
    except _error:
        pass
    else:
        raise AssertionError('invalid operation must fail')
print('Paged decode, sharing, copy-on-write, OOM atomicity, and block reclamation passed.')


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
for size in [2, 4, 8]:
    cache = PagedCache(size, 20)
    for i in range(9): cache.append("original", i, i*10)
    cache.fork("original", "branch")
    before = cache.stats()
    cache.append("branch", 9, 90)
    print({"block_size": size, "shared_prefix": before, "after_append": cache.stats()})
    assert len(cache.read("original")) == 9
    assert len(cache.read("branch")) == 10
print("Compare internal waste and block counts; this does not measure GPU throughput.")
